# 🛠️ Construct Dataset Manually (WRDS + GXZ Predictors)
Use this section if you have access to WRDS and want to reconstruct the dataset from scratch using CRSP + predictors from Dacheng Xiu’s website.

In [ ]:
# --- Imports & Setup ---
import wrds
import pandas as pd
import os
import numpy as np

# --- Connect to WRDS ---
wrds_db = wrds.Connection()

# --- Download CRSP monthly returns ---
crsp = wrds_db.raw_sql("""
    SELECT permno, date, ret, shrout, prc, altprc, vol, retx
    FROM crsp.msf
    WHERE date >= '1957-01-01'
""")

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['yyyymm'] = crsp['date'].dt.year * 100 + crsp['date'].dt.month
crsp['me'] = crsp['prc'].abs() * crsp['shrout']
crsp['logme'] = np.log(crsp['me'])

# --- Load GXZ predictors from data_share.csv ---
chars = pd.read_csv("path/to/data_share.csv")  # 🔧 Update this path
chars.columns = chars.columns.str.lower()
chars['date'] = pd.to_datetime(chars['date'], format='%Y%m%d')
chars['yyyymm'] = chars['date'].dt.year * 100 + chars['date'].dt.month

# --- Merge CRSP with predictors ---
merged = pd.merge(chars, crsp, on=['permno', 'yyyymm'], how='inner')
merged.columns = merged.columns.str.lower()
merged = merged.fillna(0)

# --- Standardize features cross-sectionally by month ---
exclude_cols = ['permno', 'yyyymm', 'date', 'ret']
feature_cols = [col for col in merged.columns if col not in exclude_cols]
merged[feature_cols] = merged.groupby('yyyymm')[feature_cols].transform(
    lambda x: (x - x.mean()) / x.std()
)
merged = merged.fillna(0)

# --- Save yearly .parquet files ---
os.makedirs("dataset_yearly_parquet", exist_ok=True)
merged['year'] = merged['yyyymm'] // 100

for year in sorted(merged['year'].unique()):
    chunk = merged[merged['year'] == year]
    filename = f"dataset_yearly_parquet/prepared_{year}.parquet"
    chunk.to_parquet(filename, index=False)
    print(f"Saved {filename} with shape {chunk.shape}")

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


# 💾 Load Full Dataset from Prebuilt Parquet Files and Run Diagnostics

In [ ]:


# Set the validation length
validation_length = 12

# Create a DataFrame for estimation periods
oos_years = range(1987, 2022)  # 1987 to 2021 inclusive
estimation_periods = pd.DataFrame({
    'oos_year': list(oos_years)
})
estimation_periods['validation_end'] = estimation_periods['oos_year'] - 1
estimation_periods['validation_start'] = estimation_periods['oos_year'] - validation_length
estimation_periods['training_start'] = 1957
estimation_periods['training_end'] = estimation_periods['validation_start'] - 1

# Select only the columns containing "training" and the other needed columns
cols = [col for col in estimation_periods.columns if 'training' in col] + ['validation_start', 'validation_end', 'oos_year']
estimation_periods = estimation_periods[cols]

# Build the visualization data by iterating over each estimation period
rows = []
for _, row in estimation_periods.iterrows():
    # Loop through each year from 1957 to 2021
    for year in range(1957, 2022):
        classification = None
        # Check conditions for classification as in the R code
        if row['training_start'] <= year <= row['training_end']:
            classification = "Training"
        elif row['validation_start'] <= year <= row['validation_end']:
            classification = "Validation"
        elif year == row['oos_year']:
            classification = "OOS"
        
        # Only add rows with a non-null classification
        if classification is not None:
            rows.append({
                'year': year,
                'oos_year': row['oos_year'],
                'classification': classification
            })

# Create the DataFrame and drop any rows with missing values (if any)
visualization_data = pd.DataFrame(rows)

# Plotting the results using Matplotlib
fig, ax = plt.subplots(figsize=(10, 6))

# Plot each classification separately to assign distinct colors
for cls in visualization_data['classification'].unique():
    subset = visualization_data[visualization_data['classification'] == cls]
    ax.scatter(subset['year'], subset['oos_year'], label=cls, s=20)  # s controls marker size

# Customize the plot to mirror the ggplot style
ax.set_title("Data classification timeline")
ax.set_xlabel("")
ax.set_ylabel("")

# Remove y-axis ticks and labels
ax.set_yticks([])
ax.tick_params(axis='y', which='both', length=0)

# Remove legend title
ax.legend(title="")

plt.tight_layout()
plt.show()